# 🧪 VitroVision — รันชุด 100 ขวด ครบในไฟล์เดียว

> ใช้รัน `sam3_growth_pipeline.py` บนชุด `20260814_batch` (100 ภาพพริกจินดา) ได้ `plant_growth_summary.csv` — **รวมทุกอย่างในนี้แล้ว**
>
> ### สิ่งที่ต้องมีก่อน
> 1. **อัปโหลด `_staging_VitroVision_colab.zip` ไว้ที่ `MyDrive/`** (โฟลเดอร์ไหนก็ได้) — ฝั่งในมี script + config + ข้อมูล 100 ภาพ
> 2. **GPU**: Runtime → Change runtime type → GPU (T4)
> 3. **Token**: วางใน **cell #2** (จาก huggingface.co → Settings → Access Tokens, ต้องเข้าถึง `facebook/sam3`)
>
> แล้วกด **Run all** — notebook จะหา zip → แตกให้เอง → แก้บั๊ก RAM ให้อัตโนมัติ → รันครบ 100 → ดาวน์โหลดผล
>
> 🔒 ไฟล์นี้เป็นไฟล์ส่วนตัว อย่า commit ขึ้น GitHub หลังใส่ token จริง

In [ ]:
# 1) ติดตั้ง dependency (รวม transformers — ใช้โหลด facebook/sam3)
!pip -q install torch torchvision transformers opencv-python pillow matplotlib pandas numpy \
    huggingface_hub openpyxl xlsxwriter tabulate
print("deps OK")

In [ ]:
# 2) วาง Hugging Face token ที่นี่ (คัดจาก huggingface.co → Settings → Access Tokens)
#    อย่า commit ไฟล์นี้ขึ้น GitHub หลังใส่ token จริง
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxx"   # ← วาง token ของคุณตรงนี้ (แทน hf_xxx...)

import os
from huggingface_hub import login
tok = os.environ.get("HF_TOKEN") or HF_TOKEN
if not tok or not tok.startswith("hf_"):
    raise SystemExit("ยังไม่ได้วาง token — แก้ HF_TOKEN = 'hf...' ใน cell นี้ แล้วรันใหม่")
login(token=tok, add_to_git_credential=False)
os.environ["HF_TOKEN"] = tok
print("HF login OK")

In [ ]:
# 3) Mount Google Drive (กด Allow ถ้าขึ้น popup)
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted")

In [ ]:
# 4) เตรียมข้อมูล: หา zip ทั่ว Drive → แตกให้อัตโนมัติ → คัด script/config → แก้บั๊ก RAM
import os, zipfile, shutil, glob
DRIVE = "/content/drive/MyDrive"

def _find_zip(root, max_depth=4):
    for d, dirs, files in os.walk(root):
        depth = d[len(root):].count(os.sep)
        if depth >= max_depth:
            dirs[:] = []
            continue
        dirs[:] = [x for x in dirs if not x.startswith('.')]
        for fn in files:
            if fn.lower().endswith('.zip') and 'vitrovision' in fn.lower():
                return os.path.join(d, fn)
    return None

def _find_dir_with_pipeline(base):
    for d, dirs, files in os.walk(base):
        if 'sam3_growth_pipeline.py' in files:
            return d
        dirs[:] = [x for x in dirs if not x.startswith('.')]
    return None

SRC = os.path.join(DRIVE, "VitroVision_colab")
if not os.path.isdir(SRC):
    zname = _find_zip(DRIVE)
    if zname is not None:
        print(f"[info] พบ zip: {zname}")
        with zipfile.ZipFile(zname) as z:
            z.extractall(os.path.dirname(zname))
        print("[ok] แตก zip เสร็จ")
    SRC = os.path.join(DRIVE, "VitroVision_colab")
    if not os.path.isdir(SRC):
        found = _find_dir_with_pipeline(DRIVE)
        SRC = found or SRC
assert os.path.isdir(SRC), ("ยังไม่พบโฟลเดอร์ VitroVision_colab — ตรวจว่า zip '_staging_VitroVision_colab.zip' อัปโหลดขึ้น Drive จริง")
print("[ok] SRC =", SRC)

# คัด script + config
shutil.copy(os.path.join(SRC, "sam3_growth_pipeline.py"), "/content/")
if not os.path.exists("/content/config.json"):
    shutil.copy(os.path.join(SRC, "config.json"), "/content/")

# ==== แก้บั๊ก RAM ให้อัตโนมัติ (เก็บ mask แค่ 6 ภาพแรก กันพัง ~51) ====
sp = "/content/sam3_growth_pipeline.py"
src = open(sp, encoding="utf-8").read()
if "KEEP_MASK_FIRST" not in src:
    src = src.replace(
        "    rows = []\n    all_masks = {}\n    progress_path",
        "    rows = []\n    all_masks = {}\n    KEEP_MASK_FIRST = 6\n    progress_path")
    src = src.replace(
        "        all_masks[name] = mbp\n        rows.append(feat)",
        "        if i <= KEEP_MASK_FIRST:\n            all_masks[name] = mbp\n        del mbp\n        rows.append(feat)")
    open(sp, "w", encoding="utf-8").write(src)
    print("[ok] auto-patch: แก้บั๊ก RAM ลง script แล้ว (KEEP_MASK_FIRST=6)")
else:
    print("[ok] script มี RAM fix อยู่แล้ว")

# เตรียมภาพ batch
batch_dir = None
for cand in [os.path.join(SRC, "20260814_batch.zip"), os.path.join(SRC, "_staging_20260814_batch.zip")]:
    if os.path.exists(cand):
        with zipfile.ZipFile(cand) as z:
            z.extractall("/content/data")
        batch_dir = "/content/data"
        print(f"[ok] แตกจาก zip: {cand}")
        break
if batch_dir is None:
    for cand in [os.path.join(SRC, "20260814_batch"), os.path.join(SRC, "data")]:
        if os.path.isdir(cand):
            shutil.copytree(cand, "/content/data", dirs_exist_ok=True)
            batch_dir = "/content/data"
            print(f"[ok] คัดลอกจากโฟลเดอร์: {cand}")
            break
if batch_dir is None:
    raise SystemExit("ไม่พบอาร์ติแฟกต์ชุด 100 — วาง zip/โฟลเดอร์ใน VitroVision_colab/ บน Drive ก่อน")

imgs = sorted(f for f in os.listdir(batch_dir) if f.lower().endswith(".jpg"))
print("ภาพ:", len(imgs), "| script:", os.path.exists("/content/sam3_growth_pipeline.py"),
      "| config:", os.path.exists("/content/config.json"))

In [ ]:
# 5) รัน pipeline (SAM3 5 prompts + ROI ขวด + verdict 3 คลาส)
import time, os
t0 = time.time()
# อยากได้การันตี segment แม่นโดยไม่มี ground truth → เซ็ต RUN_SYNTHETIC=1 (ได้ benchmark_IoU_Dice_MAE.csv)
RUN_EXTRA = "--synthetic" if os.environ.get("RUN_SYNTHETIC", "0") == "1" else ""
!python /content/sam3_growth_pipeline.py --data /content/data --out /content/results \
    --config /content/config.json {RUN_EXTRA}
print(f"RUNTIME_MIN={(time.time()-t0)/60:.1f}")

In [ ]:
# 6) บันทึกผลลง Drive + ดาวน์โหลดกลับเครื่อง
import shutil, os
from datetime import datetime
stamp = datetime.now().strftime("%Y%m%d_%H%M")
shutil.make_archive(f"/content/results_evidence_{stamp}", "zip", "/content/results")
dst = f"/content/drive/MyDrive/VitroVision_colab/results_evidence_{stamp}.zip"
shutil.copy(f"/content/results_evidence_{stamp}.zip", dst)
print("บันทึก Drive:", dst)
from google.colab import files
files.download(f"/content/results_evidence_{stamp}.zip")

for f in ["plant_growth_summary.csv", "_progress.csv"]:
    p = os.path.join("/content/results", f)
    print(("✅ " if os.path.exists(p) else "❌ "), f, f"({os.path.getsize(p)} bytes)" if os.path.exists(p) else "")

## 📥 หลังรันเสร็จ
เอา `plant_growth_summary.csv` (แตกจาก zip ที่ดาวน์โหลด/บน Drive) มาให้ — วางใน `data/processed/` ของโปรเจกต์ แล้วส่งมา ผมจะวิเคราะห์ต่อให้

## ⚠️ หมายเหตุ
- ภาพชุด 100 = พริกจินดา ชนิดเดียว 3 วันถ่าย (16/7, 2/8, 14/8) → ชุด **time-series**
- ยังไม่มี `ground_truth.csv` → ถ้าจะ validate กับมือจริง ต้องวัด manual ก่อน ดู `docs/DATA_TEMPLATES.md`